# Seasonal Agriculture Performance Analysis

**Major Project — VOIS AICTE Batch 1 (2026–2027)**

**Prepared by:** K Dharshan
**College:** Sri Venkateswara College of Engineering and Technology
**AICTE Student ID:** *[Add your AICTE Student ID here]*

---

## 1. Introduction

This project analyzes a dataset of **4,000 farm-level agricultural records** collected across
three cropping seasons (**Kharif, Rabi, Zaid**), 8 Indian states, 10 districts and 8 crops.

The dataset captures farming practices, environmental conditions, resource usage, crop yield/production,
and economic performance (cost, revenue, profit) for each farm record.

The goal of this notebook is to explore how agricultural performance changes across seasons and to
uncover meaningful, **data-driven** patterns, trends and relationships — not to make generic observations
about the whole dataset, but to answer a focused, real analytics question:

> **How does agricultural performance vary across seasons, and what factors are associated with that variation?**

## 2. Problem Statement

Agricultural activities are influenced by seasonal variations in environmental conditions, farming
practices, resource availability and market conditions. As a result, agricultural performance may differ
from one season to another.

However, raw agricultural data does not by itself explain **how** performance changes across seasons or
**what patterns** exist. The problem addressed in this project is to analyze the given agricultural
dataset and investigate seasonal differences in performance by identifying meaningful patterns, trends,
relationships and variations within the available data.

## 3. Objectives

- Explore and understand the dataset structure and variables.
- Clean and prepare the data for analysis (handle missing values, verify data types and validity).
- Examine how yield, production, revenue and profit vary across seasons.
- Identify important seasonal patterns and crop-level trends.
- Investigate relationships between environmental conditions (rainfall, temperature, humidity, soil, etc.)
  and yield.
- Compare resource usage (water, fertilizer, irrigation method) across seasons and its link to performance.
- Compare economic performance across seasons, crops and states.
- Apply descriptive statistics, correlation analysis and visualization techniques.
- Derive evidence-based insights and practical, data-backed recommendations.

## 4. Dataset Description

The dataset `seasonal_agriculture_performance_dataset.csv` contains **4,000 rows and 28 columns**.
Each row represents one farm's performance record for a given crop and season.

| Group | Columns |
|---|---|
| **Identifiers** | `Farm_ID`, `State`, `District`, `Crop`, `Season` |
| **Farm & Environment** | `Farm_Area_Hectares`, `Rainfall_mm`, `Avg_Temperature_C`, `Humidity_pct`, `Sunlight_Hours_Day`, `Soil_pH`, `Soil_Moisture_pct` |
| **Soil Nutrients** | `Nitrogen_kg_ha`, `Phosphorus_kg_ha`, `Potassium_kg_ha` |
| **Farming Practices** | `Irrigation_Method`, `Fertilizer_kg_ha`, `Pesticide_Litre_ha`, `Seed_Quality_Score` |
| **Production Outcomes** | `Yield_Tonnes_Ha`, `Production_Tonnes` |
| **Economic Outcomes** | `Market_Price_INR_Tonne`, `Total_Cost_INR`, `Revenue_INR`, `Profit_INR` |
| **Water & Risk** | `Water_Used_m3`, `Water_Efficiency_t_per_1000m3`, `Disease_Pest_Risk_pct` |

**Note on geography:** The State–District combinations in this dataset are randomly assigned for
simulation purposes and do not represent real administrative mappings (e.g. the same district name
appears under multiple states). Regional analysis in this notebook is therefore performed **at the
State level**, which is used consistently as the identifier for a farm's operating region.

## 5. Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 11

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

SEASON_ORDER = ['Kharif', 'Rabi', 'Zaid']
SEASON_COLORS = {'Kharif': '#2E7D32', 'Rabi': '#F9A825', 'Zaid': '#C62828'}

## 6. Load Dataset

In [ ]:
df = pd.read_csv('../data/seasonal_agriculture_performance_dataset.csv')
print(f"Dataset loaded successfully with shape: {df.shape}")

## 7. Dataset Overview

**Expected output:** the dataset has **4,000 rows and 28 columns**. There are no full duplicate rows,
and `Farm_ID` is unique for every record.

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.tail()

In [ ]:
df.info()

In [ ]:
df.describe(include='all').T

In [ ]:
print("Duplicate rows:", df.duplicated().sum())
print("Duplicate Farm_IDs:", df['Farm_ID'].duplicated().sum())
print("\nUnique States:", df['State'].nunique(), "->", list(df['State'].unique()))
print("Unique Districts:", df['District'].nunique())
print("Unique Crops:", df['Crop'].nunique(), "->", list(df['Crop'].unique()))
print("Unique Seasons:", df['Season'].nunique(), "->", list(df['Season'].unique()))
print("Unique Irrigation Methods:", df['Irrigation_Method'].nunique(), "->", list(df['Irrigation_Method'].unique()))

**Interpretation:** The dataset covers 8 states, 10 districts, 8 crops, 3 standard Indian cropping
seasons (Kharif, Rabi, Zaid) and 4 irrigation methods (Drip, Flood, Rainfed, Sprinkler). There are
**zero duplicate records**, confirming each row is an independent farm observation.

## 8. Data Quality Analysis

**Expected output:** Missing values were found in exactly 3 columns: `Rainfall_mm` (48 missing),
`Soil_Moisture_pct` (40 missing) and `Yield_Tonnes_Ha` (32 missing) — a total of 120 missing values
out of 112,000 cells (about **0.11%** of all data).

In [ ]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_report = pd.DataFrame({'Missing_Count': missing, 'Missing_%': missing_pct.round(2)})
missing_report[missing_report['Missing_Count'] > 0]

In [ ]:
# Check for negative values in columns that should never be negative
non_negative_cols = ['Farm_Area_Hectares', 'Rainfall_mm', 'Yield_Tonnes_Ha', 'Production_Tonnes',
                      'Water_Used_m3', 'Fertilizer_kg_ha', 'Pesticide_Litre_ha',
                      'Total_Cost_INR', 'Revenue_INR']
for c in non_negative_cols:
    print(f"{c:28s} -> negative values: {(df[c] < 0).sum():>4}  | min value: {df[c].min()}")

print("\nNote: Profit_INR is allowed to be negative (a loss-making farm). ")
print("Negative-profit records:", (df['Profit_INR'] < 0).sum(), f"({(df['Profit_INR']<0).mean()*100:.2f}% of farms)")

**Interpretation:** None of the physical/quantity columns contain invalid negative values.
`Profit_INR` is negative for **1,966 of 4,000 records (49.15%)**, which is expected and meaningful —
it represents farms that operated at a loss, not a data error.

In [ ]:
# Check for outliers in Yield (initially looks extreme) using IQR method
for c in ['Yield_Tonnes_Ha', 'Water_Efficiency_t_per_1000m3', 'Revenue_INR', 'Profit_INR']:
    q1, q3 = df[c].quantile(0.25), df[c].quantile(0.75)
    iqr = q3 - q1
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n_out = ((df[c] < lo) | (df[c] > hi)).sum()
    print(f"{c:32s}: {n_out:4d} IQR outliers ({n_out/len(df)*100:.2f}%)")

print("\nYield by crop (checking whether 'outliers' are genuine crop differences):")
print(df.groupby('Crop')['Yield_Tonnes_Ha'].agg(['mean', 'min', 'max']).round(2))

**Important finding:** `Yield_Tonnes_Ha` shows ~7.5% "outliers" by the IQR rule, but grouping by
crop shows this is **not a data error** — Sugarcane genuinely yields 38–53 t/ha on average (real-world
sugarcane yields are 60–100 t/ha), which is far higher than grain crops like Wheat, Rice or Pulses
(1–3 t/ha). These extreme values are kept in the dataset because they reflect genuine crop biology,
not incorrect entries.

In [ ]:
# Check categorical consistency (no stray spelling variants / inconsistent casing)
for c in ['State', 'District', 'Crop', 'Season', 'Irrigation_Method']:
    print(f"\n{c}: {sorted(df[c].unique())}")

**Interpretation:** All categorical values are clean and consistently spelled — no case-mismatches
or duplicate category labels (e.g. no "kharif" vs "Kharif") were found.

## 9. Data Cleaning

**Cleaning Plan**

| Issue Found | Column(s) | Method Chosen | Reason |
|---|---|---|---|
| Missing values (48 rows) | `Rainfall_mm` | Median imputation, grouped by `Crop` + `Season` | Rainfall is strongly tied to the season and crop calendar, so a group-wise median is more representative than a single global median. |
| Missing values (40 rows) | `Soil_Moisture_pct` | Median imputation, grouped by `Crop` + `Season` | Soil moisture depends on season and irrigation pattern typical of a crop; group median avoids distorting the distribution. |
| Missing values (32 rows) | `Yield_Tonnes_Ha` | Median imputation, grouped by `Crop` + `Season` | Yield varies enormously by crop (e.g. Sugarcane vs Pulses); a group-wise median preserves crop-specific yield levels instead of pulling them toward an unrealistic global average. |
| Apparent "outliers" in Yield | `Yield_Tonnes_Ha` | **Not removed** | Verified to be genuine Sugarcane yield values, not data entry errors — removing them would discard valid information. |
| Full duplicate rows | — | **None found**, no action needed | — |

Rows were **not dropped** and no values were arbitrarily capped, in line with the requirement to avoid
unnecessary or excessive data modification.

In [ ]:
df_clean = df.copy()

for col in ['Rainfall_mm', 'Soil_Moisture_pct', 'Yield_Tonnes_Ha']:
    before_missing = df_clean[col].isnull().sum()
    df_clean[col] = df_clean.groupby(['Crop', 'Season'])[col].transform(lambda x: x.fillna(x.median()))
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())  # safety net for any fully-empty group
    print(f"{col}: filled {before_missing} missing values")

# IMPORTANT: Yield imputation changes the input to downstream metrics. Recalculate these
# fields so production, revenue, profit and water efficiency remain internally consistent.
df_clean['Production_Tonnes'] = (df_clean['Farm_Area_Hectares'] * df_clean['Yield_Tonnes_Ha']).round(2)
df_clean['Revenue_INR'] = (df_clean['Production_Tonnes'] * df_clean['Market_Price_INR_Tonne']).round().astype('int64')
df_clean['Profit_INR'] = (df_clean['Revenue_INR'] - df_clean['Total_Cost_INR']).astype('int64')
df_clean['Water_Efficiency_t_per_1000m3'] = (df_clean['Production_Tonnes'] / df_clean['Water_Used_m3'] * 1000).round(3)

print("\nRemaining missing values after cleaning:", df_clean.isnull().sum().sum())

# Validation checks for derived metrics (tolerances account for documented rounding).
production_error = (df_clean['Production_Tonnes'] - (df_clean['Farm_Area_Hectares'] * df_clean['Yield_Tonnes_Ha']).round(2)).abs()
revenue_error = (df_clean['Revenue_INR'] - (df_clean['Production_Tonnes'] * df_clean['Market_Price_INR_Tonne']).round()).abs()
profit_error = (df_clean['Profit_INR'] - (df_clean['Revenue_INR'] - df_clean['Total_Cost_INR'])).abs()
water_error = (df_clean['Water_Efficiency_t_per_1000m3'] - (df_clean['Production_Tonnes'] / df_clean['Water_Used_m3'] * 1000).round(3)).abs()
print(f"Production formula mismatches: {(production_error > 0.001).sum()}")
print(f"Revenue formula mismatches: {(revenue_error > 0).sum()}")
print(f"Profit formula mismatches: {(profit_error > 0).sum()}")
print(f"Water-efficiency formula mismatches: {(water_error > 0.001).sum()}")

**Expected output:** all 120 missing values are filled and the final missing-value count is `0`.

## 10. Calculated Metrics

Before calculating the additional per-hectare metrics, the cleaning step recalculates dependent fields after yield imputation: `Production_Tonnes = Farm_Area_Hectares × Yield_Tonnes_Ha`, `Revenue_INR = Production_Tonnes × Market_Price_INR_Tonne`, `Profit_INR = Revenue_INR − Total_Cost_INR`, and `Water_Efficiency_t_per_1000m3 = Production_Tonnes / Water_Used_m3 × 1000`. Values are rounded to the precision used in the dataset. Validation checks in the cleaning cell confirm these relationships.

The following derived metrics are created:

1. **Profit Margin (%)** = `Profit_INR / Revenue_INR * 100` — shows what share of revenue is retained as profit; useful for comparing profitability independent of farm size.
2. **Revenue per Hectare** = `Revenue_INR / Farm_Area_Hectares` — normalizes revenue by farm size for fair comparison between small and large farms.
3. **Profit per Hectare** = `Profit_INR / Farm_Area_Hectares` — normalizes profit by farm size.
4. **Cost per Hectare** = `Total_Cost_INR / Farm_Area_Hectares` — shows cultivation cost intensity per unit area, useful for cost-efficiency comparisons.

Division-by-zero cases are treated as missing values for ratio metrics rather than creating infinite values. The original CSV is preserved unchanged; the corrected output is written to `data/seasonal_agriculture_performance_dataset_CLEANED.csv`.

In [ ]:
df_clean['Profit_Margin_pct'] = (df_clean['Profit_INR'] / df_clean['Revenue_INR']) * 100
df_clean['Revenue_per_Hectare'] = df_clean['Revenue_INR'] / df_clean['Farm_Area_Hectares']
df_clean['Profit_per_Hectare'] = df_clean['Profit_INR'] / df_clean['Farm_Area_Hectares']
df_clean['Cost_per_Hectare'] = df_clean['Total_Cost_INR'] / df_clean['Farm_Area_Hectares']

df_clean[['Profit_Margin_pct', 'Revenue_per_Hectare', 'Profit_per_Hectare', 'Cost_per_Hectare']].describe().round(2)

In [ ]:
# Save the cleaned + enriched dataset for reference
df_clean.to_csv('../data/seasonal_agriculture_performance_dataset_CLEANED.csv', index=False)
print("Cleaned dataset saved.")

# 11. Exploratory Data Analysis

## 11.1 Seasonal Performance Analysis

**Expected output:** Kharif has the most records (1,779), followed by Rabi (1,627) and Zaid (594).

In [ ]:
season_counts = df_clean['Season'].value_counts().reindex(SEASON_ORDER)
print(season_counts)

fig, ax = plt.subplots(figsize=(6,4))
sns.barplot(x=season_counts.index, y=season_counts.values,
            hue=season_counts.index, palette=SEASON_COLORS, legend=False, ax=ax)
ax.set_title('Number of Farm Records by Season')
ax.set_xlabel('Season'); ax.set_ylabel('Number of Records')
for i, v in enumerate(season_counts.values):
    ax.text(i, v + 20, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../images/01_farms_by_season.png', bbox_inches='tight')
plt.show()

In [ ]:
season_yield = df_clean.groupby('Season')['Yield_Tonnes_Ha'].mean().reindex(SEASON_ORDER).sort_values(ascending=False)
print("Average Yield by Season (Tonnes/Hectare):")
print(season_yield.round(3))

fig, ax = plt.subplots(figsize=(6,4))
sns.barplot(x=season_yield.index, y=season_yield.values,
            hue=season_yield.index, palette=SEASON_COLORS, legend=False, ax=ax)
ax.set_title('Average Crop Yield by Season')
ax.set_xlabel('Season'); ax.set_ylabel('Avg Yield (Tonnes/Hectare)')
for i, v in enumerate(season_yield.values):
    ax.text(i, v + 0.1, f'{v:.2f}', ha='center', fontweight='bold')
plt.tight_layout()
plt.savefig('../images/02_avg_yield_by_season.png', bbox_inches='tight')
plt.show()

**Expected output:** **Kharif records the highest average yield at 5.63 t/ha**, followed by Rabi
(5.09 t/ha) and Zaid (4.64 t/ha).

In [ ]:
season_production = df_clean.groupby('Season')['Production_Tonnes'].sum().reindex(SEASON_ORDER).sort_values(ascending=False)
print("Total Production by Season (Tonnes):")
print(season_production.round(0))

fig, ax = plt.subplots(figsize=(6,4))
sns.barplot(x=season_production.index, y=season_production.values,
            hue=season_production.index, palette=SEASON_COLORS, legend=False, ax=ax)
ax.set_title('Total Production by Season')
ax.set_xlabel('Season'); ax.set_ylabel('Total Production (Tonnes)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x/1000:.0f}K'))
plt.tight_layout()
plt.savefig('../images/03_total_production_by_season.png', bbox_inches='tight')
plt.show()

**Expected output:** **Kharif has the highest total production (82,388 tonnes)**, ahead of Rabi
(67,499 tonnes) and Zaid (23,098 tonnes) — consistent with Kharif also having the most farm records.

In [ ]:
season_revenue = df_clean.groupby('Season')['Revenue_INR'].mean().reindex(SEASON_ORDER).sort_values(ascending=False)
season_profit = df_clean.groupby('Season')['Profit_INR'].mean().reindex(SEASON_ORDER).sort_values(ascending=False)
print("Average Revenue by Season (INR):\n", season_revenue.round(0))
print("\nAverage Profit by Season (INR):\n", season_profit.round(0))

**Expected output:** **Kharif generates the highest average revenue (₹7,10,719) and the highest
average profit (₹1,78,915)**. Rabi is profitable but lower (₹87,689 avg profit). **Zaid records a
negative average profit of ₹−24,805**, making it the weakest season economically.

In [ ]:
season_water_eff = df_clean.groupby('Season')['Water_Efficiency_t_per_1000m3'].mean().reindex(SEASON_ORDER).sort_values(ascending=False)
season_water_used = df_clean.groupby('Season')['Water_Used_m3'].mean().reindex(SEASON_ORDER).sort_values(ascending=False)
print("Average Water Efficiency by Season:\n", season_water_eff.round(2))
print("\nAverage Water Used by Season (m3):\n", season_water_used.round(0))

fig, axes = plt.subplots(1, 2, figsize=(12,4))
sns.barplot(x=season_water_used.index, y=season_water_used.values, hue=season_water_used.index,
            palette=SEASON_COLORS, legend=False, ax=axes[0])
axes[0].set_title('Average Water Usage by Season'); axes[0].set_ylabel('Avg Water Used (m³)')
sns.barplot(x=season_water_eff.index, y=season_water_eff.values, hue=season_water_eff.index,
            palette=SEASON_COLORS, legend=False, ax=axes[1])
axes[1].set_title('Average Water Efficiency by Season'); axes[1].set_ylabel('Tonnes / 1000 m³')
plt.tight_layout()
plt.savefig('../images/06_water_usage_by_season.png', bbox_inches='tight')
plt.savefig('../images/07_water_efficiency_by_season.png', bbox_inches='tight')
plt.show()

**Expected output:** **Zaid uses the most water on average (6,420 m³)** despite having the fewest
records, yet achieves the **lowest water efficiency (4.41 t/1000m³)**. **Kharif is the most water-efficient
season (5.89 t/1000m³)** even though it doesn't use the least water — meaning Kharif converts water into
yield more effectively than Zaid.

In [ ]:
season_yield_std = df_clean.groupby('Season')['Yield_Tonnes_Ha'].std().reindex(SEASON_ORDER).sort_values(ascending=False)
print("Yield variability (std. deviation) by Season:")
print(season_yield_std.round(2))

**Expected output:** **Kharif shows the greatest variation in yield (std ≈ 14.39 t/ha)**, followed
by Rabi (12.83) and Zaid (11.27). This is expected because Kharif is the main season in which
high-yielding Sugarcane is grown alongside low-yielding grains, widening the spread.

In [ ]:
season_rainfall = df_clean.groupby('Season')['Rainfall_mm'].mean().reindex(SEASON_ORDER)
season_temp = df_clean.groupby('Season')['Avg_Temperature_C'].mean().reindex(SEASON_ORDER)
season_disease = df_clean.groupby('Season')['Disease_Pest_Risk_pct'].mean().reindex(SEASON_ORDER)

fig, axes = plt.subplots(1, 3, figsize=(15,4))
sns.barplot(x=season_rainfall.index, y=season_rainfall.values, hue=season_rainfall.index,
            palette=SEASON_COLORS, legend=False, ax=axes[0])
axes[0].set_title('Avg Rainfall by Season'); axes[0].set_ylabel('mm')
sns.barplot(x=season_temp.index, y=season_temp.values, hue=season_temp.index,
            palette=SEASON_COLORS, legend=False, ax=axes[1])
axes[1].set_title('Avg Temperature by Season'); axes[1].set_ylabel('°C')
sns.barplot(x=season_disease.index, y=season_disease.values, hue=season_disease.index,
            palette=SEASON_COLORS, legend=False, ax=axes[2])
axes[2].set_title('Avg Disease/Pest Risk by Season'); axes[2].set_ylabel('%')
plt.tight_layout()
plt.savefig('../images/04_avg_rainfall_by_season.png', bbox_inches='tight')
plt.savefig('../images/05_avg_temperature_by_season.png', bbox_inches='tight')
plt.savefig('../images/17_disease_pest_by_season.png', bbox_inches='tight')
plt.show()

print("Rainfall (mm):\n", season_rainfall.round(1))
print("\nTemperature (C):\n", season_temp.round(1))
print("\nDisease/Pest Risk (%):\n", season_disease.round(1))

**Expected output:** Kharif is the wettest season (852 mm avg rainfall) and Rabi is the coolest
(23.5°C avg). Zaid is the hottest (31.0°C avg) and driest (299 mm avg). **Kharif also has the highest
average disease/pest risk (54.5%)**, consistent with its high rainfall and humidity favoring pest and
disease pressure, while Zaid has the lowest risk (38.2%).

## 11.2 Crop Performance Analysis

**Expected output:** Sugarcane has by far the highest average yield.

In [ ]:
crop_yield = df_clean.groupby('Crop')['Yield_Tonnes_Ha'].mean().sort_values(ascending=False)
crop_production = df_clean.groupby('Crop')['Production_Tonnes'].sum().sort_values(ascending=False)
print("Average Yield by Crop:\n", crop_yield.round(2))
print("\nTotal Production by Crop:\n", crop_production.round(0))

fig, ax = plt.subplots(figsize=(8,4.5))
sns.barplot(x=crop_yield.index, y=crop_yield.values, hue=crop_yield.index,
            palette='YlGn_r', legend=False, ax=ax)
ax.set_yscale('log')
ax.set_title('Average Yield by Crop (log scale)')
ax.set_ylabel('Avg Yield (Tonnes/Hectare)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../images/08_avg_yield_by_crop.png', bbox_inches='tight')
plt.show()

**Expected output:** **Sugarcane has the highest average yield at 46.93 t/ha**, over 17x higher
than the next crop (Maize, 2.72 t/ha). Sugarcane is also the highest in total production
(119,686 tonnes), followed by Rice (13,309 t) and Maize (12,001 t).

In [ ]:
crop_profit = df_clean.groupby('Crop')['Profit_INR'].mean().sort_values(ascending=False)
crop_margin = df_clean.groupby('Crop')['Profit_Margin_pct'].mean().sort_values(ascending=False)
print("Average Profit by Crop (INR):\n", crop_profit.round(0))
print("\nAverage Profit Margin by Crop (%):\n", crop_margin.round(1))

fig, ax = plt.subplots(figsize=(8,4.5))
colors = ['#2E7D32' if v >= 0 else '#C62828' for v in crop_profit.values]
sns.barplot(x=crop_profit.index, y=crop_profit.values, hue=crop_profit.index,
            palette=colors, legend=False, ax=ax)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_title('Average Profit by Crop'); ax.set_ylabel('Avg Profit (INR)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x/1000:.0f}K'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../images/09_avg_profit_by_crop.png', bbox_inches='tight')
plt.show()

**Expected output:** **Sugarcane is the most profitable crop on average (₹8,17,188)**, followed
by Chilli (₹7,50,878). By profit *margin*, **Chilli has the best margin at 33.8%**, ahead of Sugarcane
(27.3%). In contrast, **Rice, Wheat, Maize and Pulses show negative average profit and negative margins**,
meaning the average farm growing these crops in this dataset is not covering its costs.

In [ ]:
crop_season_yield = df_clean.pivot_table(index='Crop', columns='Season', values='Yield_Tonnes_Ha', aggfunc='mean')[SEASON_ORDER]
crop_season_yield.round(2)

In [ ]:
fig, ax = plt.subplots(figsize=(7,5.5))
sns.heatmap(crop_season_yield, annot=True, fmt='.2f', cmap='YlGn', ax=ax, cbar_kws={'label': 'Avg Yield (t/ha)'})
ax.set_title('Average Yield: Crop × Season Heatmap')
plt.tight_layout()
plt.savefig('../images/14_crop_season_heatmap.png', bbox_inches='tight')
plt.show()

**Expected output:** Every crop in this dataset yields best in **Kharif**, followed by Rabi, then
Zaid — this ordering is consistent across all 8 crops. Sugarcane shows the largest absolute seasonal
swing (53.46 t/ha in Kharif vs 38.42 t/ha in Zaid), while grain crops like Pulses show a smaller but
still consistent decline (1.04 → 0.65 t/ha).

## 11.3 Environmental Analysis

We examine the relationship between environmental conditions and yield. Because yield scales differ
enormously by crop (Sugarcane vs grains), a **raw dataset-wide correlation** can be misleading. We look
at both the overall correlation and the **average within-crop correlation** (which removes the crop-scale
effect) for a fairer picture.

In [ ]:
env_cols = ['Rainfall_mm', 'Avg_Temperature_C', 'Humidity_pct', 'Sunlight_Hours_Day', 'Soil_pH', 'Soil_Moisture_pct']

overall_corr = df_clean[env_cols + ['Yield_Tonnes_Ha']].corr()['Yield_Tonnes_Ha'].drop('Yield_Tonnes_Ha').sort_values(ascending=False)
print("Overall correlation with Yield (whole dataset):")
print(overall_corr.round(3))

within_crop_corr = {}
for col in env_cols:
    within_crop_corr[col] = df_clean.groupby('Crop').apply(lambda g: g[col].corr(g['Yield_Tonnes_Ha'])).mean()
within_crop_corr = pd.Series(within_crop_corr).sort_values(key=abs, ascending=False)
print("\nAverage within-crop correlation with Yield (removes crop-scale effect):")
print(within_crop_corr.round(3))

**Expected output:** the overall (whole-dataset) correlations between environmental factors and
yield are all close to zero (|r| < 0.03), because the huge yield gap between Sugarcane and grain crops
swamps any environmental signal. Once we look **within each crop**, clearer (though still moderate)
patterns emerge: **Rainfall (avg r ≈ 0.22)**, **Soil Moisture (≈ 0.18)** and **Humidity (≈ 0.16)** show
the strongest positive association with yield, while **Sunlight Hours (≈ −0.12)** shows a mild negative
association. Soil pH and Temperature show almost no linear relationship with yield.

**These are correlations/observed patterns only — not proof of causation.**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))
sample = df_clean.sample(min(1500, len(df_clean)), random_state=42)

sns.scatterplot(data=sample, x='Rainfall_mm', y='Yield_Tonnes_Ha', hue='Season', alpha=0.6, ax=axes[0], palette=SEASON_COLORS)
axes[0].set_yscale('log')
axes[0].set_title('Rainfall vs Yield (log scale)')

sns.scatterplot(data=sample, x='Avg_Temperature_C', y='Yield_Tonnes_Ha', hue='Season', alpha=0.6, ax=axes[1], palette=SEASON_COLORS)
axes[1].set_yscale('log')
axes[1].set_title('Temperature vs Yield (log scale)')

plt.tight_layout()
plt.savefig('../images/11_rainfall_vs_yield.png', bbox_inches='tight')
plt.savefig('../images/12_temperature_vs_yield.png', bbox_inches='tight')
plt.show()

## 11.4 Resource Usage Analysis

In [ ]:
fert_yield_corr = df_clean['Fertilizer_kg_ha'].corr(df_clean['Yield_Tonnes_Ha'])
water_yield_corr = df_clean['Water_Used_m3'].corr(df_clean['Yield_Tonnes_Ha'])
print(f"Correlation: Fertilizer usage vs Yield  = {fert_yield_corr:.4f}")
print(f"Correlation: Water used vs Yield        = {water_yield_corr:.4f}")

irrigation_yield = df_clean.groupby('Irrigation_Method')['Yield_Tonnes_Ha'].mean().sort_values(ascending=False)
irrigation_eff = df_clean.groupby('Irrigation_Method')['Water_Efficiency_t_per_1000m3'].mean().sort_values(ascending=False)
print("\nAverage Yield by Irrigation Method:\n", irrigation_yield.round(2))
print("\nAverage Water Efficiency by Irrigation Method:\n", irrigation_eff.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12,4.5))
sns.barplot(x=irrigation_yield.index, y=irrigation_yield.values, hue=irrigation_yield.index,
            palette='Blues_r', legend=False, ax=axes[0])
axes[0].set_title('Average Yield by Irrigation Method'); axes[0].set_ylabel('Avg Yield (t/ha)')
sns.barplot(x=irrigation_eff.index, y=irrigation_eff.values, hue=irrigation_eff.index,
            palette='Blues_r', legend=False, ax=axes[1])
axes[1].set_title('Average Water Efficiency by Irrigation Method'); axes[1].set_ylabel('t / 1000 m³')
plt.tight_layout()
plt.savefig('../images/16_irrigation_vs_yield.png', bbox_inches='tight')
plt.show()

**Expected output:** Fertilizer usage shows **virtually no linear correlation with yield**
(r ≈ -0.0004) — more fertilizer does not automatically mean more yield in this dataset. Water used
shows a **moderate positive correlation with yield (r ≈ 0.39)**, so higher water usage is *associated*
with higher yield, but not perfectly — efficiency also matters.

By irrigation method: **Drip irrigation gives the highest average yield (6.67 t/ha)**, ahead of
Sprinkler (5.16), Flood (4.86) and Rainfed (4.60). However, **Rainfed farming shows the highest water
efficiency (7.56 t/1000m³)** because it uses no supplemental water at all, while **Flood irrigation is
the least water-efficient method (3.44 t/1000m³)** despite using large volumes of water. This means
higher water usage does **not always** translate to proportionally higher yield — Drip irrigation
achieves the best yield with far more efficient water use than Flood irrigation.

## 11.5 Economic Performance Analysis

In [ ]:
season_revenue_full = df_clean.groupby('Season')[['Revenue_INR','Profit_INR']].mean().reindex(SEASON_ORDER)
crop_profit_full = df_clean.groupby('Crop')[['Revenue_INR','Profit_INR','Profit_Margin_pct']].mean().sort_values('Profit_INR', ascending=False)
state_profit = df_clean.groupby('State')['Profit_INR'].mean().sort_values(ascending=False)
state_revenue = df_clean.groupby('State')['Revenue_INR'].mean().sort_values(ascending=False)

print("Revenue & Profit by Season:\n", season_revenue_full.round(0))
print("\nProfit & Margin by Crop:\n", crop_profit_full.round(1))
print("\nAverage Profit by State:\n", state_profit.round(0))

fig, ax = plt.subplots(figsize=(8,4.5))
sns.barplot(x=state_profit.index, y=state_profit.values, hue=state_profit.index,
            palette='crest', legend=False, ax=ax)
ax.set_title('Average Profit by State'); ax.set_ylabel('Avg Profit (INR)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, p: f'{x/1000:.0f}K'))
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('../images/10_top_states_by_profit.png', bbox_inches='tight')
plt.show()

In [ ]:
prod_profit_corr = df_clean['Production_Tonnes'].corr(df_clean['Profit_INR'])
neg_profit_pct = (df_clean['Profit_INR'] < 0).mean() * 100
print(f"Correlation: Production vs Profit = {prod_profit_corr:.4f}")
print(f"Percentage of farms operating at a loss = {neg_profit_pct:.2f}%")

**Expected output:** **Kharif is the economically strongest season** with the highest average
revenue (₹7,10,719) and profit (₹1,78,915); **Zaid is the weakest**, averaging a loss of ₹−24,805 per
farm. By crop, **Sugarcane generates the highest average profit (₹8,17,188)** and **Chilli has the best
profit margin (33.8%)**. By state, **Punjab has the highest average profit (₹1,36,026)**, followed
closely by Maharashtra (₹1,35,429); **Andhra Pradesh is the lowest (₹73,454)**.

Production and profit are **moderately positively correlated (r ≈ 0.55)** — higher production tends to
come with higher profit, but this is not guaranteed: **49.15% of all farms in the dataset recorded a net
loss**, showing that high output alone does not guarantee profitability once costs are factored in.

## 11.6 Regional Analysis

*(State is used as the regional unit — see the note on District/State mapping in Section 4.)*

In [ ]:
state_summary = df_clean.groupby('State').agg(
    Avg_Yield=('Yield_Tonnes_Ha','mean'),
    Total_Production=('Production_Tonnes','sum'),
    Avg_Revenue=('Revenue_INR','mean'),
    Avg_Profit=('Profit_INR','mean'),
    Avg_Water_Efficiency=('Water_Efficiency_t_per_1000m3','mean')
).sort_values('Avg_Profit', ascending=False).round(2)
state_summary

**Expected output:** **Punjab is the top-performing state overall** — highest average yield
(6.12 t/ha) and highest average profit (₹1,36,026). **Andhra Pradesh is the lowest-performing state**
on both yield (4.63 t/ha) and profit (₹73,454). Karnataka and Gujarat also perform strongly on yield
and water efficiency, suggesting these states may have more favorable growing conditions or practices
represented in the dataset.

## 12. Statistical Analysis

Summary statistics (mean, median, min, max, std) for the key performance indicators, both overall and
grouped by Season, Crop, State and Irrigation Method.

In [ ]:
key_cols = ['Yield_Tonnes_Ha','Production_Tonnes','Revenue_INR','Profit_INR','Water_Used_m3','Water_Efficiency_t_per_1000m3']
df_clean[key_cols].agg(['mean','median','min','max','std']).round(2).T

In [ ]:
print("Grouped statistics by Season:")
df_clean.groupby('Season')[key_cols].agg(['mean','std']).round(2)

In [ ]:
print("Grouped statistics by Irrigation Method:")
df_clean.groupby('Irrigation_Method')[key_cols].agg(['mean','std']).round(2)

**Interpretation:** Profit has the largest relative spread (std >> mean) of all key indicators,
reflecting the mix of highly profitable crops (Sugarcane, Chilli) with loss-making ones (Rice, Wheat,
Maize, Pulses) in the same dataset. Yield and Water Efficiency are also right-skewed — driven by the
Sugarcane sub-population — while Production and Revenue are more moderately spread.

## 13. Correlation Analysis

In [ ]:
num_cols = ['Rainfall_mm','Avg_Temperature_C','Humidity_pct','Sunlight_Hours_Day','Soil_pH','Soil_Moisture_pct',
            'Nitrogen_kg_ha','Phosphorus_kg_ha','Potassium_kg_ha','Fertilizer_kg_ha','Pesticide_Litre_ha',
            'Seed_Quality_Score','Yield_Tonnes_Ha','Water_Used_m3','Water_Efficiency_t_per_1000m3',
            'Revenue_INR','Profit_INR']

corr_matrix = df_clean[num_cols].corr()

fig, ax = plt.subplots(figsize=(11,9))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, ax=ax, square=True, linewidths=0.3)
ax.set_title('Correlation Heatmap of Numeric Variables')
plt.xticks(rotation=45, ha='right'); plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('../images/15_correlation_heatmap.png', bbox_inches='tight')
plt.show()

print("Strongest correlations with Yield_Tonnes_Ha:")
print(corr_matrix['Yield_Tonnes_Ha'].sort_values(ascending=False).round(3))

**Expected output:** `Water_Efficiency_t_per_1000m3` is very strongly correlated with `Yield_Tonnes_Ha`
(r ≈ 0.92) — expected, since efficiency is derived from yield and water used. `Profit_INR` and
`Water_Used_m3` show moderate positive correlations with yield (r ≈ 0.49 and 0.39 respectively).
Fertilizer, pesticide, seed quality and soil-nutrient columns show **negligible correlation** with
yield in this dataset, once again because crop type — not input intensity — is the dominant driver of
yield differences.

## 14. Key Insights

1. **Kharif is the best-performing season overall** — highest average yield (5.63 t/ha), highest total
   production (82,388 t), highest average revenue (₹7,10,719) and highest average profit (₹1,78,915).
2. **Zaid is the weakest season economically**, with an average **loss** of ₹24,805 per farm, despite
   using the most water on average (6,420 m³) — the lowest return on water of the three seasons.
3. **Sugarcane dominates yield and total output**, averaging **46.93 t/ha** — over 17× the next-highest
   crop (Maize, 2.72 t/ha) — and contributing 119,686 of the dataset's total tonnes produced.
4. **Chilli has the best profit margin (33.8%)**, narrowly ahead of Sugarcane (27.3%), making it the
   most cost-efficient crop per rupee of revenue, even though its total profit is lower than Sugarcane's.
5. **Four of the eight crops — Rice, Wheat, Maize and Pulses — show a negative average profit margin**,
   with Rice the worst (−99.4%), indicating these crops are, on average, not covering their production
   costs in this dataset.
6. **Every crop yields best in Kharif and worst in Zaid**, a consistent seasonal ranking across all 8
   crops (see the Crop × Season heatmap).
7. **Drip irrigation delivers both the highest average yield (6.67 t/ha)** among irrigation methods and
   strong water efficiency (6.27 t/1000m³), while **Flood irrigation is the least water-efficient
   (3.44 t/1000m³)** despite comparable water use.
8. **Fertilizer usage shows almost no correlation with yield (r ≈ -0.0004)**, suggesting yield in this
   dataset is driven far more by crop type and season than by fertilizer quantity alone.
9. **49.15% of all farms recorded a net loss (negative profit)**, and production is only moderately
   correlated with profit (r ≈ 0.55) — higher output does not guarantee profitability.
10. **Punjab is the top-performing state** (avg yield 6.12 t/ha, avg profit ₹1,36,026), while **Andhra
    Pradesh is the lowest-performing state** on both yield (4.63 t/ha) and profit (₹73,454).
11. Within individual crops, **rainfall (avg r ≈ 0.22) and soil moisture (avg r ≈ 0.18)** show the
    strongest positive association with yield among the environmental factors studied — though the
    dataset-wide correlation is masked by crop-scale differences.
12. **Kharif also carries the highest average disease/pest risk (54.5%)**, consistent with its higher
    rainfall and humidity, versus Zaid's lowest risk (38.2%).

## 15. Recommendations

**Finding:** Zaid season is loss-making on average (₹−24,805) despite the highest water usage.
**Recommendation:** Prioritize Zaid-season cultivation for only the most profitable, water-efficient
crop-irrigation combinations (e.g. Drip-irrigated Chilli or Sugarcane), and discourage high-water,
low-margin crops (Rice, Wheat) in Zaid.
**Expected Benefit:** Reduced water waste and fewer loss-making farms during the driest, hottest season.

---

**Finding:** Flood irrigation is the least water-efficient method (3.44 t/1000m³) even though Drip
irrigation achieves higher yield with better efficiency.
**Recommendation:** Encourage a phased shift from Flood to Drip/Sprinkler irrigation, especially for
high-value crops.
**Expected Benefit:** Higher yield per unit of water, lower water stress in dry seasons like Zaid.

---

**Finding:** Rice, Wheat, Maize and Pulses show negative average profit margins.
**Recommendation:** Review the cost structure (input cost vs market price) for these four crops and
consider crop-mix diversification toward higher-margin crops such as Chilli where local soil/climate
conditions allow.
**Expected Benefit:** Improved farm-level profitability without necessarily increasing production area.

---

**Finding:** Fertilizer usage shows no measurable relationship with yield in this dataset.
**Recommendation:** Investigate whether fertilizer is being over-applied without yield benefit; consider
soil-testing-based, crop-specific fertilizer recommendations instead of uniform application.
**Expected Benefit:** Potential input-cost savings without any yield loss.

---

**Finding:** Punjab and Maharashtra are the top profit-performing states; Andhra Pradesh is the lowest.
**Recommendation:** Study the crop mix, irrigation methods and seasonal calendar used in Punjab and
Maharashtra as potential best-practice benchmarks for lower-performing states.
**Expected Benefit:** Knowledge transfer that could raise average profitability in weaker-performing
regions.

---

**Finding:** Kharif has the highest average disease/pest risk (54.5%).
**Recommendation:** Strengthen pest and disease monitoring and early-warning systems specifically
during the Kharif season.
**Expected Benefit:** Reduced crop loss and more stable yields in the season that already contributes
the most to total production and profit.

## 16. Conclusion

This project analyzed 4,000 farm-level records spanning three cropping seasons, eight crops and eight
states to understand how agricultural performance varies seasonally. After cleaning a small number of
missing values (120 cells, ~0.11% of the dataset) using crop-and-season-aware median imputation, the
analysis found clear and consistent seasonal patterns: **Kharif is the strongest season** across yield,
production, revenue and profit, while **Zaid is the weakest**, generating an average loss despite the
highest average water usage. Crop performance varies enormously — **Sugarcane dominates yield and
profit**, while **Rice, Wheat, Maize and Pulses operate at a loss on average**. Environmental factors
such as rainfall and soil moisture show a modest positive association with yield once crop-type effects
are accounted for, while fertilizer usage shows little to no measurable relationship with yield in this
dataset. Resource-usage analysis showed that **Drip irrigation is both higher-yielding and more
water-efficient than Flood irrigation**, and that **higher water usage does not automatically translate
into better returns** — Zaid's high water use with low efficiency is the clearest example. Regionally,
**Punjab and Maharashtra emerged as the strongest-performing states**, while **Andhra Pradesh lagged**
on both yield and profit. Overall, this analysis demonstrates how structured data analytics — cleaning,
grouping, correlation analysis and visualization — can convert a raw agricultural dataset into
actionable, evidence-based insight for seasonal planning, irrigation strategy, crop selection and
regional benchmarking.

## 17. Future Scope

The current project is a descriptive/diagnostic data analytics exercise. The following are **realistic
future extensions**, clearly distinct from the analysis already completed in this notebook:

1. **Machine learning-based crop yield prediction** using regression/ensemble models trained on
   environmental and farm-practice features.
2. **A crop recommendation system** that suggests the most suitable and profitable crop for a given
   season, soil and region.
3. **Real-time weather data integration** (via a weather API) to make the analysis dynamic rather than
   based on a static historical dataset.
4. **An interactive Power BI / Tableau dashboard** for stakeholders to explore seasonal trends without
   needing to run the notebook.
5. **IoT sensor integration** for real-time soil moisture, temperature and irrigation monitoring at the
   farm level.
6. **Disease and pest risk prediction models** built on the `Disease_Pest_Risk_pct` column and
   environmental drivers.
7. **A farmer-facing decision-support system / mobile application** that surfaces these seasonal
   insights and recommendations in a simple, actionable interface.

These are proposed directions for future work and were **not implemented as part of this project.**